In [23]:
# Tic-Tac-Toe: Player1 (X) vs Player2 (O)
import numpy as np
import random

def print_board(board, show_reference=False):
    if show_reference:
        print("\nIndex Reference (0-8):")
        print(" 0 | 1 | 2 ")
        print("---+---+---")
        print(" 3 | 4 | 5 ")
        print("---+---+---")
        print(" 6 | 7 | 8 ")
        print()

    print(f"{board[0]}|{board[1]}|{board[2]}")
    print("-+-+-")
    print(f"{board[3]}|{board[4]}|{board[5]}")
    print("-+-+-")
    print(f"{board[6]}|{board[7]}|{board[8]}")
    print()

def check_winner(board, player):
    win_states = [
        [0,1,2],[3,4,5],[6,7,8],  # rows
        [0,3,6],[1,4,7],[2,5,8],  # cols
        [0,4,8],[2,4,6]           # diagonals
    ]
    return any(all(board[i] == player for i in line) for line in win_states)

def tic_tac_toe():
    board = [' '] * 9
    current_player = 'X'  # Player1 starts

    # Index reference
    print_board(board, show_reference=True)

    for turn in range(9):
        move = int(input(f"Player {current_player}, enter your move (0-8): "))
        while move not in range(9) or board[move] != ' ':
            move = int(input("Invalid move. Try again (0-8): "))

        board[move] = current_player
        print_board(board)

        if check_winner(board, current_player):
            print(f"Player {current_player} wins!")
            return

        # Switch player
        current_player = 'O' if current_player == 'X' else 'X'

    print("It's a draw!")

tic_tac_toe()



Index Reference (0-8):
 0 | 1 | 2 
---+---+---
 3 | 4 | 5 
---+---+---
 6 | 7 | 8 

 | | 
-+-+-
 | | 
-+-+-
 | | 



KeyboardInterrupt: Interrupted by user

In [41]:
import random

class TicTacToeEnv:
    def __init__(self):
        self.reset()

    def reset(self):
        self.board = [' '] * 9
        return tuple(self.board)

    def available_actions(self):
        return [i for i, spot in enumerate(self.board) if spot == ' ']

    def step(self, action, player):
        # 执行动作
        self.board[action] = player

        # 胜利奖励：立即获胜最高奖励
        if self.check_winner(player):
            return tuple(self.board), 1.0, True

        # 平局奖励
        if ' ' not in self.board:
            return tuple(self.board), 0.0, True

        # 默认步数惩罚：鼓励更快结束游戏
        reward = -0.05

        # 中心格奖励
        if action == 4:
            reward += 0.2

        # 二连奖励：只奖励自己形成的二连
        win_states = [
            [0,1,2],[3,4,5],[6,7,8],
            [0,3,6],[1,4,7],[2,5,8],
            [0,4,8],[2,4,6]
        ]
        for line in win_states:
            values = [self.board[i] for i in line]
            if values.count(player) == 2 and values.count(' ') == 1:
                empty_index = line[values.index(' ')]
                # 如果这个空格能让自己获胜 → 奖励更高
                temp_board = self.board.copy()
                temp_board[empty_index] = player
                if self.check_winner_on_board(temp_board, player):
                    reward += 0.5  # 逼近胜利的奖励
                else:
                    reward += 0.3  # 普通二连奖励

        return tuple(self.board), reward, False

    def check_winner(self, player):
        win_states = [
            [0,1,2],[3,4,5],[6,7,8],
            [0,3,6],[1,4,7],[2,5,8],
            [0,4,8],[2,4,6]
        ]
        return any(all(self.board[i] == player for i in line) for line in win_states)

    def check_winner_on_board(self, board, player):
        win_states = [
            [0,1,2],[3,4,5],[6,7,8],
            [0,3,6],[1,4,7],[2,5,8],
            [0,4,8],[2,4,6]
        ]
        return any(all(board[i] == player for i in line) for line in win_states)

class QLearningAgent:
    def __init__(self, alpha=0.1, gamma=0.9, epsilon=0.2):
        self.q_table = {}
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon

    def get_q(self, state, action):
        return self.q_table.get((state, action), 0.0)

    def choose_action(self, state, actions):
        if random.uniform(0,1) < self.epsilon:
            return random.choice(actions)
        q_values = [self.get_q(state,a) for a in actions]
        max_q = max(q_values)
        return actions[q_values.index(max_q)]

    def learn(self, state, action, reward, next_state, next_actions, done):
        old_q = self.get_q(state, action)
        target = reward if done else reward + self.gamma * max([self.get_q(next_state,a) for a in next_actions], default=0)
        self.q_table[(state, action)] = old_q + self.alpha * (target - old_q)


# Training Process
env = TicTacToeEnv()
agent = QLearningAgent(alpha=0.2, gamma=0.95, epsilon=1.0)

episodes = 50000
for ep in range(episodes):
    agent.epsilon = max(0.01, agent.epsilon * 0.999)  # 探索率逐渐降低
    state = env.reset()
    done = False
    current_player = 'X'
    while not done:
        actions = env.available_actions()
        action = agent.choose_action(state, actions)
        next_state, reward, done = env.step(action, current_player)
        agent.learn(state, action, reward, next_state, env.available_actions(), done)
        state = next_state
        current_player = 'O' if current_player == 'X' else 'X'

print("✅ Training complete!")



✅ Training complete!


In [42]:
# Verify
wins, losses, draws = 0, 0, 0
for _ in range(1000):
    state = env.reset()
    done = False
    current_player = 'X'
    while not done:
        actions = env.available_actions()
        action = agent.choose_action(state, actions)
        state, reward, done = env.step(action, current_player)
        current_player = 'O' if current_player == 'X' else 'X'
    if reward == 1:
        wins += 1
    elif reward == -1:
        losses += 1
    else:
        draws += 1

print(f"AI win rate: {wins}, Loss: {losses}, Draw: {draws}")

AI win rate: 1000, Loss: 0, Draw: 0


In [44]:
def play_vs_ai(agent):
    env = TicTacToeEnv()
    state = env.reset()
    done = False

    # Player select go first or second
    player_symbol = input("X for First, O for Second:").upper()
    while player_symbol not in ['X','O']:
        player_symbol = input("Invalid input! Please enter X or O:").upper()
    ai_symbol = 'O' if player_symbol == 'X' else 'X'

    current_player = 'X'  # start by X
    print_board(env.board, show_reference=True)

    while not done:
        if current_player == player_symbol:  # Human player round
            move = int(input(f"Player {player_symbol}, Please enter your placement 0-8:"))
            while move not in env.available_actions():
                move = int(input("Invalid placement, please enter 0-8:"))
            state, reward, done = env.step(move, player_symbol)
        else:  # AI round
            actions = env.available_actions()
            action = agent.choose_action(state, actions)
            state, reward, done = env.step(action, ai_symbol)

        print_board(env.board)

        if done:
            if reward == 1 and current_player == ai_symbol:
                print("🤖 AI win!")
            elif reward == 1 and current_player == player_symbol:
                print("🎉 You win!")
            elif reward == -1 and current_player == ai_symbol:
                print("🎉 You win!")
            elif reward == -1 and current_player == player_symbol:
                print("🤖 AI win!")
            else:
                print("🤝 draw!")
            break

        # Switch Player
        current_player = 'O' if current_player == 'X' else 'X'

# run
play_vs_ai(agent)



X for First, O for Second:x

Index Reference (0-8):
 0 | 1 | 2 
---+---+---
 3 | 4 | 5 
---+---+---
 6 | 7 | 8 

 | | 
-+-+-
 | | 
-+-+-
 | | 

Player X, Please enter your placement 0-8:2
 | |X
-+-+-
 | | 
-+-+-
 | | 

 | |X
-+-+-
 |O| 
-+-+-
 | | 

Player X, Please enter your placement 0-8:1
 |X|X
-+-+-
 |O| 
-+-+-
 | | 

O|X|X
-+-+-
 |O| 
-+-+-
 | | 

Player X, Please enter your placement 0-8:5
O|X|X
-+-+-
 |O|X
-+-+-
 | | 

O|X|X
-+-+-
 |O|X
-+-+-
 | |O

🤖 AI win!
